In [0]:
# ============================================================
# NOTEBOOK   : bronze_ingestion
# PURPOSE    : Read all 9 CSV files from ADLS Gen2 and write
#              them as Delta tables into bronze schema
#              NO transformations happen here
#              Raw data is preserved exactly as it came in
# LAYER      : Bronze
# SOURCE     : kyc-aml-dev container (asadev)
# DESTINATION: fincompliance_catalog.bronze
# ============================================================

In [0]:
%run ../configs/config




In [0]:

## Authenricate & Set Catalog
authenticate_spark(spark)

## Setting the active catalog
set_catalog(spark)

In [0]:
from pyspark.sql.functions import current_timestamp, lit



In [0]:
## Bronze Ingestion Loop
# ============================================================
# BRONZE INGESTION LOOP
# Reads all 9 CSV files from ADLS and writes them
# as Delta tables into fincompliance_catalog.bronze
# ============================================================

for table_name, file_path in RAW_PATHS.items():
    print(f"\n{'='*60}")
    print(f"Processing : {table_name}")
    print(f"Source : {file_path}")

    df = spark.read.option("header","true").option("inferSchema","true").csv(file_path)
    df = df\
        .withColumn("ingestion_timestamp", current_timestamp()) \
        .withColumn("source_file", lit(file_path.split("/")[-1]))

    row_count = df.count()
    print(f"Rows found : {row_count:,}")
    print(f"Columns : {df.columns}")

    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{BRONZE_DB}.{table_name}")

    print(f"Status : written to {BRONZE_DB}.{table_name}")
    print(f"\n{'='*60}")
    print("Bronze ingestion complete!")
    print(f"All nine tables written to {BRONZE_DB}")
    





In [0]:
# ============================================================
# BRONZE VERIFICATION
# Confirms all 9 tables are written correctly
# Run this after the ingestion loop completes
# ============================================================


print("=" * 60)
print("BRONZE LAYER VERIFICATION")
print(f"Catalog : {BRONZE_DB}")
print("=" * 60)

total_rows = 0

for table in RAW_PATHS.keys():
    try:
        count = spark.table(f"{BRONZE_DB}.{table}").count()
        total_rows += count
        print(f"{table} : {count} rows")
    except Exception as e:
        print(f"{table} : FAILED — {str(e)}")

print("=" * 60)
print(f"  {'TOTAL':<25} : {total_rows:>10,} rows")
print("=" * 60)

print("\nVerifying audit columns customers table:")
df_check = spark.table(f"{BRONZE_DB}.customers")
columns = df_check.columns

if "ingestion_timestamp" in columns:
    print("ingestion_timestamp column exists")
else:
    print("ingestion_timestamp column MISSING")

if "source_file" in columns:
    print("source_file column exists")
else:
    print("source_file column MISSING")

print("\nSample data from bronze.customers (3 rows):")
spark.table(f"{BRONZE_DB}.customers").show(3, truncate=True)

print("\n Bronze verification complete!")
print("Bronze layer is ready for Silver transformations.")